# GAN

In [6]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

In [4]:
img_shape = (28,28,1)
z_dim = 100
row_num = 8
col_num = 8
batch_size = 64
epoch_num = 10
learning_rate = 0.0001

In [7]:
(X_train,y_train),(X_test,y_test) = tf.keras.datasets.mnist.load_data()
X_train = X_train.astype('float32')/255.0
y_train = np.reshape(X_train,(-1,28,28,1))

# Discriminator 모델 (판별자)
- 이미지가 가짜인지 진짜인지 판단하는 모델

In [11]:
# 특징 추출
i = tf.keras.Input(shape=img_shape)

out = tf.keras.layers.Conv2D(16, 3, 2, padding='same')(i)
out = tf.keras.layers.Conv2D(32, 3, 2, padding='same')(out)
out = tf.keras.layers.Conv2D(64, 3, 2, padding='same')(out)

# 1차원으로 변환
out = tf.keras.layers.Flatten()(out)

# 정규화
out = tf.keras.layers.BatchNormalization()(out)

# Dense
out = tf.keras.layers.Dense(1024, activation='tanh')(out)
out = tf.keras.layers.Dense(1, activation='sigmoid')(out)

d_model = tf.keras.Model(inputs=[i], outputs=[out])

d_model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 28, 28, 1)]       0         
                                                                 
 conv2d_6 (Conv2D)           (None, 14, 14, 16)        160       
                                                                 
 conv2d_7 (Conv2D)           (None, 7, 7, 32)          4640      
                                                                 
 conv2d_8 (Conv2D)           (None, 4, 4, 64)          18496     
                                                                 
 flatten_1 (Flatten)         (None, 1024)              0         
                                                                 
 batch_normalization (BatchN  (None, 1024)             4096      
 ormalization)                                                   
                                                             

# Generator 모델 (가짜를 생성)
- 랜덤 노이즈를 입력받아서 가짜 숫자 이미지를 만드는 모델(위조지폐범)

In [14]:
# 100차원의 랜덤 벡터를 입력받아서 이미지 생성

i = tf.keras.Input(shape=(z_dim,))

out = tf.keras.layers.Dense(1024, activation='tanh')(i)
out = tf.keras.layers.Dense(7 * 7 * 32, activation='tanh')(out)

out = tf.keras.layers.BatchNormalization()(out)

# (7, 7, 32) 형태로 변환
out = tf.keras.layers.Reshape((7, 7, 32))(out)

# 업샘플링
out = tf.keras.layers.Conv2DTranspose(
    16, 3, strides=2, padding='same'
)(out)

out = tf.keras.layers.Conv2DTranspose(
    1, 3, strides=2, padding='same'
)(out)

# 출력 범위 0~1
out = tf.keras.layers.Activation('sigmoid')(out)

g_model = tf.keras.Model(inputs=i, outputs=out)

g_model.summary()

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense_4 (Dense)             (None, 1024)              103424    
                                                                 
 dense_5 (Dense)             (None, 1568)              1607200   
                                                                 
 batch_normalization_2 (Batc  (None, 1568)             6272      
 hNormalization)                                                 
                                                                 
 reshape_1 (Reshape)         (None, 7, 7, 32)          0         
                                                                 
 conv2d_transpose_2 (Conv2DT  (None, 14, 14, 16)       4624      
 ranspose)                                                 

# GAN 학습
- Minist 이미지 준비
- 판별자 모델과 생성자 모델 반복 학습
    - 가짜 이미지를 가짜라고 학습
    - 진짜 이미지를 진짜라고 학습
    - Generator는 가짜를 진짜처럼 보이게 학습
- 학습 중 이미지는를 영상을 저장

In [17]:
opt = tf.keras.optimizers.Adam(learning_rate)
d_loss = tf.keras.losses.BinaryCrossentropy()

In [22]:
fcc=cv2.VideoWriter_fourcc(*'DIVX')
out=cv2.VideoWriter('hjk_gan_mnist.avi', fcc, 1.0, (28*row_num, 28*col_num))

batch_count =X_train.shape[0]//batch_size

for e in range(epoch_num):
    for _ in range(batch_count):

        # 가짜 이미지 생성
        z = np.random.uniform(-1.0, 1.0, (batch_size, z_dim))
        f_img = g_model.predict(z) # g_model(z, training=True)
        f_label = np.zeros((batch_size, 1))

        # Discrimiator가 가짜이미지를 보고 예측
        with tf.GradientTape() as tape:
            pred = d_model(f_img)
            loss = d_loss(f_label, pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        opt.apply_gradients(zip(grad, vars))

        batch_num=np.random.randint(0,  X_train.shape[0],  size=batch_size)
        r_img = X_train[batch_num]
        r_label = np.ones((batch_size, 1))

        # Discrimiator를 이용해서 진짜 지폐를 1이라고 학습
        with tf.GradientTape() as tape:
            pred = d_model(r_img)
            loss = d_loss(r_label, pred)
        vars = d_model.trainable_variables
        grad = tape.gradient(loss, vars)
        opt.apply_gradients(zip(grad, vars))

        # 가짜 지폐는 진짜 (1)이라고 Discrimiator에게 속임
        # Gemerator 목표 : 가짜 이미지를 만들었지만 Discrimiator가 1이라고 예측하게 만들기
        with tf.GradientTape() as tape:
            f_img = g_model(z)
            pred = d_model(f_img)
            loss = d_loss(r_label, pred)
        vars = g_model.trainable_variables
        grad = tape.gradient(loss, vars)
        opt.apply_gradients(zip(grad, vars))


        sample_img = np.zeros((28*row_num, 28 * col_num))
        f_img = np.resize(f_img, (row_num, col_num, 28, 28))

        for i in range(row_num):
            for j in range(col_num):
                sample_img[i * 28:i * 28 +28, j * 28:j * 28 +28] = f_img[i, j, :, :]
        sample_img = np.uint8(sample_img * 255.)
        sample_img = cv2.applyColorMap(sample_img, cv2.COLORMAP_HOT)
        out.write(sample_img)

    print(e, "완료")
out.release()

2/2 [==============================] - 0s 2ms/step
0 완료
2/2 [==============================] - 0s 5ms/step
1 완료
2/2 [==============================] - 0s 4ms/step
2 완료
2/2 [==============================] - 0s 3ms/step
3 완료
2/2 [==============================] - 0s 2ms/step
4 완료
2/2 [==============================] - 0s 3ms/step
5 완료
2/2 [==============================] - 0s 4ms/step
6 완료
2/2 [==============================] - 0s 2ms/step
7 완료
2/2 [==============================] - 0s 4ms/step
8 완료
2/2 [==============================] - 0s 5ms/step
9 완료
